# OOF (Out-of-Fold) 기반 수요예측 보정 모델

## 방법론 개요

**OOF(Out-of-Fold) 예측**은 K-Fold 교차검증 방식과 동일하다.
학습 데이터를 K개 폴드로 나누어 K번 학습하되,
각 폴드에서 검증 구간은 학습에 사용되지 않은 모델이 예측한다.

```
Fold 1:  [학습][학습][학습][학습][ 검증 ]  → 검증 예측 + Test 예측₁
Fold 2:  [학습][학습][학습][ 검증 ][학습]  → 검증 예측 + Test 예측₂
Fold 3:  [학습][학습][ 검증 ][학습][학습]  → 검증 예측 + Test 예측₃
Fold 4:  [학습][ 검증 ][학습][학습][학습]  → 검증 예측 + Test 예측₄
Fold 5:  [ 검증 ][학습][학습][학습][학습]  → 검증 예측 + Test 예측₅

최종 Test 예측 = 평균(Test 예측₁, Test 예측₂, ..., Test 예측₅)
```

**시계열 적용 시 주의**: 미래 정보 누출(leakage) 방지를 위해  
`KFold(shuffle=True)` 대신 `TimeSeriesSplit`을 사용한다.  
폴드가 시간 순서를 따르므로 각 폴드의 학습셋 크기가 점진적으로 증가한다.

**Base Models**: LightGBM, GradientBoosting, PatchTST (Nie et al., 2023)  
**Naive Drift**: 결정론적 모델로 OOF와 별도 계산, 최종 앙상블에 포함

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble        import GradientBoostingRegressor
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import mean_squared_error, mean_absolute_error

import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

In [ ]:
DATA_PATH     = 'data_weekly_260120.csv'
TARGET_COL    = 'Com_LME_Ni_Cash'

VAL_START     = pd.to_datetime('2025-08-04')
VAL_END       = pd.to_datetime('2025-10-20')
TEST_START    = pd.to_datetime('2025-10-27')
TEST_END      = pd.to_datetime('2026-01-12')

N_SPLITS      = 5       # OOF K-Fold 수
RANDOM_STATE  = 42
BASELINE_RMSE = 406.80  # sparta2_advanced.ipynb 기준

# ── PatchTST ─────────────────────────────────────────────────────────────
SEQ_LEN      = 26   # 26주 ≈ 6개월
PATCH_LEN    = 4    # 4주 ≈ 1개월  →  n_patches = (26-4)//2 + 1 = 12
STRIDE       = 2
D_MODEL      = 32
N_HEAD       = 4
N_LAYERS     = 2
DROPOUT      = 0.1
LR           = 5e-4
BATCH_SIZE   = 32
OOF_EPOCHS   = 40
FINAL_EPOCHS = 100
PATIENCE     = 15

## 1. 데이터 로딩 · 피처 엔지니어링 · 분할

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['dt']).set_index('dt').sort_index()
print(f'기간: {df.index[0].date()} ~ {df.index[-1].date()}  |  shape: {df.shape}')

# 피처 엔지니어링 (sparta2_advanced.ipynb 동일 기준)
feat_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feat_cols].shift(1)
y = df[TARGET_COL]

log_ret = np.log(y / y.shift(1))
for w in [4, 8, 12, 26]:
    X[f'RV_{w}w']      = log_ret.shift(1).rolling(w).std() * np.sqrt(52)
for w in [4, 12, 26]:
    X[f'ROC_{w}w']     = y.shift(1).pct_change(w)
for w in [4, 26]:
    mu, sig            = y.shift(1).rolling(w).mean(), y.shift(1).rolling(w).std()
    X[f'zscore_{w}w']  = (y.shift(1) - mu) / (sig + 1e-8)
for lag in [1, 2, 3]:
    X[f'ret_lag_{lag}'] = log_ret.shift(lag)

valid = X.notna().all(axis=1) & y.notna()
X, y  = X[valid], y[valid]
N_VARS = X.shape[1]
print(f'유효 샘플: {len(X)}  |  피처: {N_VARS}')

tr = X.index < VAL_START
va = (X.index >= VAL_START)  & (X.index <= VAL_END)
te = (X.index >= TEST_START) & (X.index <= TEST_END)

X_train, y_train = X[tr], y[tr]
X_val,   y_val   = X[va], y[va]
X_test,  y_test  = X[te], y[te]

for tag, d in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {tag:5s}: {len(d):4d}개  ({d.index[0].date()} ~ {d.index[-1].date()})')

## 2. 공통 유틸리티

In [ ]:
def eval_metrics(y_true, y_pred, name=''):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    print(f'  [{name:40s}]  RMSE={rmse:7.2f}  MAE={mae:7.2f}  MAPE={mape:.2f}%')
    return dict(model=name, rmse=rmse, mae=mae, mape=mape)


def naive_drift(y_series, n_steps):
    """y_{t+k} = y_t + k * (y_t - y_{t-1})"""
    last, drift = y_series.iloc[-1], y_series.iloc[-1] - y_series.iloc[-2]
    return np.array([last + (k + 1) * drift for k in range(n_steps)])


def make_sequences(X_arr, seq_len):
    """슬라이딩 윈도우: X_seqs[k] = X_arr[k : k+seq_len]"""
    return np.stack([X_arr[i:i + seq_len] for i in range(len(X_arr) - seq_len)])

## 3. PatchTST 모델 정의

**Nie et al. (2023)** Channel-Independence PatchTST.  
`seq_len=26, patch_len=4, stride=2` → `n_patches=12`

In [ ]:
class PatchTST(nn.Module):
    """
    PatchTST: A Time Series is Worth 64 Words (Nie et al., 2023)
    Channel-Independence 방식: 각 변수를 독립적으로 Transformer 처리 후 채널 평균 집계.
    """
    def __init__(self, n_vars, seq_len,
                 patch_len=4, stride=2,
                 d_model=32, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        n_patches       = (seq_len - patch_len) // stride + 1
        self.patch_len  = patch_len
        self.stride     = stride
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_enc    = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_enc, std=0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.dropout  = nn.Dropout(dropout)
        self.head     = nn.Linear(d_model * n_patches, 1)

    def forward(self, x):
        B, L, C = x.shape
        x   = x.permute(0, 2, 1).unfold(2, self.patch_len, self.stride)
        n_p = x.shape[2]
        x   = x.reshape(B * C, n_p, self.patch_len)
        x   = self.dropout(self.patch_proj(x) + self.pos_enc)
        x   = self.encoder(x)
        x   = x.reshape(B, C, -1).mean(dim=1)
        return self.head(x).squeeze(-1)


def _train_patchtst(model, X_tr, y_tr, epochs, X_val=None, y_val=None, patience=None):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    ds  = TensorDataset(torch.FloatTensor(X_tr).to(DEVICE),
                        torch.FloatTensor(y_tr).to(DEVICE))
    dl  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    use_es = (X_val is not None) and (patience is not None)
    best_loss, best_state, cnt = float('inf'), None, 0

    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            nn.MSELoss()(model(xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if use_es:
            model.eval()
            with torch.no_grad():
                v = nn.MSELoss()(
                    model(torch.FloatTensor(X_val).to(DEVICE)),
                    torch.FloatTensor(y_val).to(DEVICE)
                ).item()
            if v < best_loss:
                best_loss  = v
                best_state = {k: w.cpu().clone() for k, w in model.state_dict().items()}
                cnt        = 0
            else:
                cnt += 1
                if cnt >= patience:
                    break

    if use_es and best_state:
        model.load_state_dict(best_state)
    return model


print(f'PatchTST  n_patches={(SEQ_LEN-PATCH_LEN)//STRIDE+1}  d_model={D_MODEL}')

## 4. OOF 구현

### 핵심 구조 (참고: sklearn KFold OOF 패턴)

```python
oof_preds  = np.zeros(len(X_train))   # 학습셋 전체에 대한 OOF 예측
test_preds = np.zeros(len(X_test))    # K개 모델 예측의 평균

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    model.fit(X_train[tr_idx], y_train[tr_idx])
    oof_preds[val_idx] = model.predict(X_train[val_idx])          # OOF
    test_preds        += model.predict(X_test) / N_SPLITS          # 평균 누적
```

### 시계열 OOF 적용 원칙
- `TimeSeriesSplit`: 과거→미래 방향 고정 (시간적 leakage 차단)
- DL 모델(PatchTST): fold_train에만 `StandardScaler` fit
- 시퀀스 컨텍스트: Test 예측 시 fold_train 끝부분을 컨텍스트로 사용

In [ ]:
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)
X_tr_arr = X_train.values
y_tr_arr = y_train.values
N_TR     = len(X_train)
N_VAL    = len(X_val)
N_TEST   = len(X_test)

# ── OOF 예측 배열 (학습셋 전체 커버) ────────────────────────────────────
oof_gb   = np.zeros(N_TR)
oof_lgb  = np.zeros(N_TR)
oof_ptst = np.zeros(N_TR)
oof_mask = np.zeros(N_TR, dtype=bool)

# ── Test / Val 예측 누적 배열 (K개 모델 평균) ─────────────────────────
val_gb   = np.zeros(N_VAL);  test_gb   = np.zeros(N_TEST)
val_lgb  = np.zeros(N_VAL);  test_lgb  = np.zeros(N_TEST)
val_ptst = np.zeros(N_VAL);  test_ptst = np.zeros(N_TEST)

print(f'OOF 시작  (K={N_SPLITS}, TimeSeriesSplit)\n')

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_tr_arr)):
    n_tr, n_val = len(tr_idx), len(val_idx)
    print(f'  Fold {fold+1}/{N_SPLITS}  '
          f'train={n_tr}  val={n_val}  '
          f'[{X_train.index[val_idx[0]].date()} ~ {X_train.index[val_idx[-1]].date()}]')

    X_fold_tr  = X_tr_arr[tr_idx]
    X_fold_val = X_tr_arr[val_idx]
    y_fold_tr  = y_tr_arr[tr_idx]

    # ── GradientBoosting ────────────────────────────────────────────────
    gb = GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE
    ).fit(X_fold_tr, y_fold_tr)

    oof_gb[val_idx]  = gb.predict(X_fold_val)              # OOF
    val_gb          += gb.predict(X_val.values)  / N_SPLITS  # 평균 누적
    test_gb         += gb.predict(X_test.values) / N_SPLITS

    # ── LightGBM ────────────────────────────────────────────────────────
    lgbm = lgb.LGBMRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=RANDOM_STATE, verbose=-1
    ).fit(X_fold_tr, y_fold_tr)

    oof_lgb[val_idx] = lgbm.predict(X_fold_val)              # OOF
    val_lgb         += lgbm.predict(X_val.values)  / N_SPLITS
    test_lgb        += lgbm.predict(X_test.values) / N_SPLITS

    # ── PatchTST (fold-specific scaler) ─────────────────────────────────
    if n_tr > SEQ_LEN:
        xs = StandardScaler().fit(X_fold_tr)
        ys = StandardScaler().fit(y_fold_tr.reshape(-1, 1))

        # OOF: fold_train + fold_val 결합 시퀀스
        # X_seqs[k].context = combined[k:k+SEQ_LEN]  target = combined[k+SEQ_LEN]
        # fold_val 타겟 시작: k = n_tr - SEQ_LEN  (n_tr_seqs 개 이후)
        X_comb_s = xs.transform(np.vstack([X_fold_tr, X_fold_val]))
        y_comb_s = ys.transform(
            np.concatenate([y_fold_tr, y_tr_arr[val_idx]]).reshape(-1, 1)
        ).ravel()

        X_seqs_oof = make_sequences(X_comb_s, SEQ_LEN)
        y_seqs_oof = y_comb_s[SEQ_LEN:]
        n_tr_seqs  = n_tr - SEQ_LEN

        model_oof = PatchTST(N_VARS, SEQ_LEN, PATCH_LEN, STRIDE,
                             D_MODEL, N_HEAD, N_LAYERS, DROPOUT)
        _train_patchtst(model_oof,
                        X_seqs_oof[:n_tr_seqs], y_seqs_oof[:n_tr_seqs],
                        OOF_EPOCHS)

        model_oof.eval()
        with torch.no_grad():
            pred_s = model_oof(
                torch.FloatTensor(X_seqs_oof[n_tr_seqs:]).to(DEVICE)
            ).cpu().numpy()
        oof_ptst[val_idx] = ys.inverse_transform(pred_s.reshape(-1, 1)).ravel()

        # Val / Test 예측: fold_train 끝부분을 컨텍스트로 활용
        # context (SEQ_LEN) + val + test 결합 → 시퀀스 생성
        ctx_X_s  = xs.transform(X_fold_tr[-SEQ_LEN:])
        val_X_s  = xs.transform(X_val.values)
        test_X_s = xs.transform(X_test.values)
        X_pred_s = np.vstack([ctx_X_s, val_X_s, test_X_s])
        X_seqs_pred = make_sequences(X_pred_s, SEQ_LEN)  # n_val + n_test 개

        model_oof.eval()
        with torch.no_grad():
            all_pred_s = model_oof(
                torch.FloatTensor(X_seqs_pred).to(DEVICE)
            ).cpu().numpy()
        all_pred = ys.inverse_transform(all_pred_s.reshape(-1, 1)).ravel()

        val_ptst  += all_pred[:N_VAL]  / N_SPLITS
        test_ptst += all_pred[N_VAL:]  / N_SPLITS

    else:
        # 학습 샘플 부족: GB OOF 예측으로 대체
        oof_ptst[val_idx] = oof_gb[val_idx]
        val_ptst  += val_gb  / N_SPLITS
        test_ptst += test_gb / N_SPLITS

    oof_mask[val_idx] = True

print(f'\nOOF 완료  |  유효 샘플: {oof_mask.sum()} / {N_TR}')

In [ ]:
# ── OOF 내부 성능 (학습셋 교차검증, Out-of-Sample) ───────────────────────
y_oof = y_tr_arr[oof_mask]

print('=== OOF 내부 성능 (K-Fold Out-of-Sample 예측) ===')
for arr, tag in [(oof_gb,   'OOF GradientBoosting'),
                 (oof_lgb,  'OOF LightGBM'),
                 (oof_ptst, 'OOF PatchTST')]:
    eval_metrics(y_oof, arr[oof_mask], tag)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, arr, tag in zip(axes,
                        [oof_gb, oof_lgb, oof_ptst],
                        ['GradientBoosting', 'LightGBM', 'PatchTST']):
    ax.plot(y_oof,          color='black', linewidth=1, alpha=0.7, label='실제값')
    ax.plot(arr[oof_mask],  alpha=0.8,                             label='OOF 예측')
    ax.set_title(f'OOF — {tag}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('OOF 인덱스')
    ax.set_ylabel('Nickel Price ($/ton)')
    ax.grid(alpha=0.3)

plt.suptitle('OOF 예측 vs 실제값 (학습셋 내 K-Fold 교차검증)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 최종 예측

- **ML/DL 모델**: K개 폴드 예측의 평균 (`test_preds += fold_pred / N_SPLITS`)
- **Naive Drift**: 결정론적 모델 — OOF와 무관하게 마지막 학습 가격 기준 계산
- **앙상블 가중치**: Validation 기반 Grid Search로 결정

In [ ]:
# Naive Drift (단일 모델, OOF 불필요)
nd_val  = naive_drift(y_train, N_VAL)
nd_test = naive_drift(y_val,   N_TEST)

# ── Validation 기반 앙상블 가중치 최적화 ─────────────────────────────────
best_val_rmse, best_w = float('inf'), None

for w_nd in np.arange(0.5, 1.01, 0.05):
    for w_gb in np.arange(0.0, 1.0 - w_nd + 0.01, 0.05):
        for w_lgb in np.arange(0.0, 1.0 - w_nd - w_gb + 0.01, 0.05):
            w_ptst = round(1.0 - w_nd - w_gb - w_lgb, 6)
            if w_ptst < -1e-6:
                continue
            w_ptst = max(w_ptst, 0.0)
            pred_val = w_nd * nd_val + w_gb * val_gb + w_lgb * val_lgb + w_ptst * val_ptst
            rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
            if rmse_val < best_val_rmse:
                best_val_rmse = rmse_val
                best_w        = (w_nd, w_gb, w_lgb, w_ptst)

w_nd, w_gb, w_lgb, w_ptst = best_w
print(f'=== Validation 최적 가중치 ===')
print(f'  Naive Drift   : {w_nd:.2f}')
print(f'  OOF GB (avg)  : {w_gb:.2f}')
print(f'  OOF LGB (avg) : {w_lgb:.2f}')
print(f'  OOF PatchTST  : {w_ptst:.2f}')
print(f'  Val RMSE      : {best_val_rmse:.2f}')

# ── 최종 예측 ────────────────────────────────────────────────────────────
final_val  = w_nd * nd_val  + w_gb * val_gb  + w_lgb * val_lgb  + w_ptst * val_ptst
final_test = w_nd * nd_test + w_gb * test_gb + w_lgb * test_lgb + w_ptst * test_ptst

## 6. 성능 비교

In [ ]:
results = []

print('=== Val Set 성능 ===')
eval_metrics(y_val, nd_val,                      'Naive Drift')
eval_metrics(y_val, 0.8*nd_val + 0.2*val_gb,    'Hybrid 0.8N+0.2GB (baseline 구조)')
eval_metrics(y_val, final_val,                   'OOF 앙상블 (최적 가중치)')

print('\n=== Test Set 성능 ===')
results.append(eval_metrics(y_test, nd_test,                       'Naive Drift'))
results.append(eval_metrics(y_test, test_gb,                       'OOF GB (K fold avg)'))
results.append(eval_metrics(y_test, test_lgb,                      'OOF LGB (K fold avg)'))
results.append(eval_metrics(y_test, test_ptst,                     'OOF PatchTST (K fold avg)'))
results.append(eval_metrics(y_test, 0.8*nd_test + 0.2*test_gb,    'Hybrid 0.8N+0.2GB (baseline 구조)'))
results.append(eval_metrics(y_test, final_test,                    'OOF 앙상블 (최적 가중치)'))

df_res = pd.DataFrame(results)
df_res['delta'] = df_res['rmse'] - BASELINE_RMSE
df_res['비고']   = df_res['delta'].apply(
    lambda x: f'▲ {abs(x):.2f} 개선' if x < 0 else f'▼ {x:.2f} 악화'
)
df_res = df_res.set_index('model')

print('\n' + '=' * 75)
print(f'  최종 성능 비교 (Test)     기준선 RMSE = {BASELINE_RMSE}')
print('=' * 75)
print(df_res[['rmse', 'mae', 'mape', 'delta', '비고']].to_string())
print('=' * 75)
best = df_res['rmse'].idxmin()
print(f'\n  ★ 최고 성능: {best}  (RMSE={df_res.loc[best, "rmse"]:.2f})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, idx, y_true, preds in [
    (axes[0], y_val.index,  y_val.values,
     [('실제값',      y_val.values,              'k-o', 2, 5),
      ('Naive Drift', nd_val,                    'b--', 1, 0),
      ('OOF GB',      val_gb,                    'c--', 1, 0),
      ('OOF LGB',     val_lgb,                   'g--', 1, 0),
      ('OOF PatchTST',val_ptst,                  'm-^', 1.5, 4),
      ('OOF 앙상블',  final_val,                 'r-o', 2, 5)]),
    (axes[1], y_test.index, y_test.values,
     [('실제값',      y_test.values,             'k-o', 2, 5),
      ('Naive Drift', nd_test,                   'b--', 1, 0),
      ('OOF GB',      test_gb,                   'c--', 1, 0),
      ('OOF LGB',     test_lgb,                  'g--', 1, 0),
      ('OOF PatchTST',test_ptst,                 'm-^', 1.5, 4),
      ('OOF 앙상블',  final_test,                'r-o', 2, 5)]),
]:
    for label, vals, ls, lw, ms in preds:
        ax.plot(idx, vals, ls, label=label, linewidth=lw,
                markersize=ms, alpha=(0.7 if '--' in ls else 1.0))
    title = 'Validation Set' if ax is axes[0] else 'Test Set'
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, ncol=3)
    ax.set_ylabel('Nickel Price ($/ton)')
    ax.grid(alpha=0.3)

axes[1].set_xlabel('날짜')
plt.suptitle('OOF 앙상블 (K fold 평균) vs 베이스라인 — 니켈 현물가격',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 결론

### OOF 구현 요약

| 항목 | 구현 방법 |
|------|----------|
| 폴드 분할 | `TimeSeriesSplit(K=5)` — 시간 방향 고정 |
| OOF 예측 | 각 fold_val을 해당 폴드 모델로 예측 (Out-of-sample) |
| **Test 최종 예측** | **K개 폴드 모델 예측의 평균** (`+= pred / K`) |
| DL 스케일러 | fold_train에만 fit → fold_val/test에 transform |
| 가중치 결정 | Validation 기반 Grid Search |

### 일반 KFold vs TimeSeriesSplit

| 구분 | 일반 KFold | TimeSeriesSplit |
|------|-----------|----------------|
| fold 순서 | 랜덤 셔플 가능 | 과거→미래 고정 |
| 학습셋 크기 | 균등 | 점진적 증가 |
| 시계열 leakage | 발생 가능 | 없음 |
| 적용 대상 | 독립 샘플 | 시계열 데이터 |